# Compressing Linear Layers with Comprexx

This notebook demonstrates three ways to compress models with large Linear layers:

1. **Low-rank decomposition (SVD)**: reduces parameter count by factorizing weight matrices
2. **Weight-only INT4 quantization**: reduces model storage size (weights are dequantized at runtime, so latency stays the same)
3. **Dynamic INT8 quantization**: reduces both size and latency by quantizing weights and activations at runtime

We use a feedforward network with large Linear layers so the effects are visible.

Install: `pip install comprexx`

In [1]:
import copy

import torch
import torch.nn as nn

import comprexx as cx

## 1. Define and profile the model

Two feedforward blocks (256 -> 1024 -> 256), similar to the MLP blocks inside a transformer.

In [2]:
class FeedForwardNet(nn.Module):
    def __init__(self, d_model=256, d_ff=1024, num_blocks=2, num_classes=10):
        super().__init__()
        blocks = []
        for _ in range(num_blocks):
            blocks.extend([
                nn.Linear(d_model, d_ff),
                nn.GELU(),
                nn.Linear(d_ff, d_model),
                nn.LayerNorm(d_model),
            ])
        self.blocks = nn.Sequential(*blocks)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.blocks(x)
        x = x.mean(dim=1)
        return self.classifier(x)

model = FeedForwardNet()
model.eval()

input_shape = (1, 32, 256)  # (batch, seq_len, d_model)
profile = cx.analyze(model, input_shape)
print(profile.summary())

Model: FeedForwardNet
  Architecture: unknown
  Parameters:   1,054,730 (1.1M)
  Trainable:    1,054,730
  FLOPs:        2,104,842 (0.00 GFLOPs)
  Size:         4.02 MB
  Layers:       9 (5 compressible)


## 2. Low-rank decomposition (SVD)

Factorizes each Linear layer's weight matrix into two smaller matrices using truncated SVD. We keep 90% of the spectral energy. This reduces the actual parameter count and FLOPs.

In [3]:
pipeline_lr = cx.Pipeline([
    cx.stages.LowRankDecomposition(mode="energy", energy_threshold=0.9),
])

result_lr = pipeline_lr.run(copy.deepcopy(model), input_shape=input_shape)
print(result_lr.report.summary())

# Verify output is close to original
with torch.no_grad():
    x = torch.randn(*input_shape)
    diff = (model(x) - result_lr.model(x)).abs().max()
    print(f"\nMax abs diff after SVD: {diff:.6f}")

Compression Report: FeedForwardNet
  Total duration: 0.07s
  Stages:         1
  Compression:    1.05x
  Size reduction: 4.7%
  FLOPs reduction:4.7%

Stage: low_rank_decomposition (low_rank_svd)
  Duration:    0.07s
  Params:      1,054,730 -> 1,005,668
  Size:        4.22 MB -> 4.02 MB (4.7% reduction)
  FLOPs:       2,104,842 -> 2,006,718 (4.7% reduction)
  Notes:       Decomposed 5 Linear layer(s) via truncated SVD (mode=energy).


Max abs diff after SVD: 0.234347


## 3. Weight-only INT4 quantization

Quantizes weights to 4 bits with group-wise symmetric scaling. This is a storage-focused technique: the reported size reduction reflects the theoretical packed size at 4 bits per weight. At runtime, weights are dequantized back to float32, so inference latency does not change. Real speedups require hardware with native INT4 matmul support.

In [4]:
pipeline_w4 = cx.Pipeline([
    cx.stages.WeightOnlyQuant(bits=4, group_size=64, symmetric=True),
])

result_w4 = pipeline_w4.run(copy.deepcopy(model), input_shape=input_shape)
print(result_w4.report.summary())

Compression Report: FeedForwardNet
  Total duration: 0.01s
  Stages:         1
  Compression:    7.37x
  Size reduction: 86.4%
  FLOPs reduction:0.0%

Stage: weight_only_quant (weight_only_int4)
  Duration:    0.00s
  Params:      1,054,730 -> 1,054,730
  Size:        4.22 MB -> 0.57 MB (86.4% reduction)
  FLOPs:       2,104,842 -> 2,104,842 (0.0% reduction)
  Notes:       Quantized 1,051,136 weights to INT4 (group_size=64, symmetric=True) across 5 layer(s).; Theoretical packed size: 559.4 KB (vs 4120.0 KB dense fp32).



## 4. Dynamic INT8 quantization

Quantizes Linear weights to INT8 and computes activations in INT8 at runtime. Unlike weight-only quantization, this uses PyTorch's quantized kernels, so it reduces both model size and can improve inference latency on CPU.

In [5]:
pipeline_ptq = cx.Pipeline([
    cx.stages.PTQDynamic(),
])

result_ptq = pipeline_ptq.run(copy.deepcopy(model), input_shape=input_shape)
print(result_ptq.report.summary())

Compression Report: FeedForwardNet
  Total duration: 0.01s
  Stages:         1
  Compression:    1.98x
  Size reduction: 49.6%
  FLOPs reduction:0.0%

Stage: ptq_dynamic (ptq_dynamic_int8)
  Duration:    0.01s
  Params:      1,054,730 -> 1,054,730
  Size:        4.22 MB -> 2.13 MB (49.6% reduction)
  FLOPs:       2,104,842 -> 2,104,842 (0.0% reduction)
  Notes:       Dynamic quantization applied to Linear and LSTM layers.



## 5. Benchmark: SVD vs baseline

Low-rank decomposition changes the model architecture (fewer parameters, fewer FLOPs), so it can affect latency. On small models the difference is within noise; the gains show up on larger layers.

In [6]:
cmp_lr = cx.compare_benchmarks(
    model, result_lr.model,
    input_shape=input_shape,
    warmup=10,
    iters=100,
)
print(cmp_lr.summary())

Benchmark Comparison (cpu)
  Baseline:   0.204 ms  (4895.4 ips)
  Compressed: 0.192 ms  (5213.6 ips)
  Speedup:    1.07x  (+6.1% latency, +6.5% throughput)


## 6. Summary

| Technique | Size reduction | Parameter reduction | FLOPs reduction | Latency change |
|-----------|---------------|---------------------|-----------------|----------------|
| Low-rank SVD | Moderate | Yes | Yes | Depends on layer sizes |
| Weight-only INT4 | Large (theoretical) | None | None | None (fake-quant) |
| Dynamic INT8 | ~50% | None | None (reported) | Possible on CPU |